# Implicit Q-Learning on HalfCheetah-v5

**Telman Nuruzov — theory, numerical example, notebook narrative.**
**Branch:** `feature/notebook-presentation` · **Issue:** [#4](https://github.com/MedvAx-AI/iql-halfcheetah/issues/4)

This notebook explains offline RL and IQL, works one transition by hand, and imports shared code from `iql_project`. It does **not** re-implement the learner in cells.

| Component | Status |
|---|---|
| Offline RL + IQL narrative | Done (Telman Nuruzov) |
| Manual expectile / Q-mask / capped-weight example | Done (Telman Nuruzov) |
| Environment + Minari loader | Done (Nikita Shankin, on `main`) |
| IQL networks / train / checkpoint | Done (Leo Vesin, on `main`) |
| Evaluation, plots, demo videos | Done (Ruslan Nasibullin, on `main`) |
| Measured 100k-update results | In `docs/evidence/person-4/` + [release](https://github.com/MedvAx-AI/iql-halfcheetah/releases/tag/person4-evaluation-100k) |
| Fresh Colab `Run all` | Automated below; recorded evidence in `docs/COLAB_ACCEPTANCE.md` |

Shared contracts: `IQL_5_Person_Project_Plan.md`, `docs/INTERFACES.md`, `docs/EVALUATION.md`, `configs/halfcheetah.toml`.

## 1. Introduction — offline RL and project goal

### Online vs offline

In **online** RL the agent interacts with the environment and can explore.

In **offline** RL the agent receives a **fixed dataset** of transitions

$$(s, a, r, s', \text{terminated}, \text{truncated})$$

collected beforehand. During training it cannot query the environment for new actions. Standard Q-learning is then dangerous: a $\max$ over actions can exploit out-of-distribution actions that look artificially good.

### What this project demonstrates

1. Load Minari `mujoco/halfcheetah/medium-v0`.
2. Train **Implicit Q-Learning (IQL)** on that fixed data only.
3. Evaluate in Gymnasium **HalfCheetah-v5**.
4. Show a demo video and report raw returns.

| Team member | Responsibility |
|---|---|
| Aleksandr Medvedev | Team lead, integration and Colab |
| Nikita Shankin | Environment and offline dataset |
| Leo Vesin | IQL implementation and training |
| Ruslan Nasibullin | Evaluation, plots and videos |
| Telman Nuruzov | Theory, notebook and presentation |

## 2. Setup — one-click Colab and a locked project kernel

In Google Colab select **Run all** on a CPU runtime. The next cell clones the public
repository, installs uv 0.8.22, and creates the locked Python **3.11** environment.
No Drive mounting, GPU or runtime downgrade is needed. Colab may use Python 3.12
and preload other NumPy versions; `%%project` runs subsequent cells in **one persistent
project kernel** and forwards text, plots and video to this notebook. Variables
such as `dataset` and `config` live in that kernel. Errors stop execution.

Run all downloads the public Minari dataset and the 64 MB checkpoint archive,
trains a fresh **1,000-update smoke run**, replays the original three **100k-update**
checkpoints twice, records a new MP4 and runs the test suite. It does not retrain
the three 100k experiments or the default 500k schedule. Allow several minutes.

Locally launch with `uv sync --frozen --extra dev --extra notebook`, then
`uv run --frozen jupyter lab notebooks/iql_halfcheetah.ipynb` from the repository root.
The existing locked interpreter is used locally. Set `MINARI_DATASETS_PATH` for a
custom cache. For an exact revision in Colab, set `IQL_GIT_REF` before setup; otherwise
the repository is cloned at current main. The final report records the actual SHA.

In [ ]:
import os
import platform
import runpy
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/iql-halfcheetah")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/MedvAx-AI/iql-halfcheetah.git", str(ROOT)],
            check=True,
        )
    if os.environ.get("IQL_GIT_REF"):
        subprocess.run(["git", "fetch", "origin"], cwd=ROOT, check=True)
        subprocess.run(["git", "checkout", os.environ["IQL_GIT_REF"]], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"], check=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--python",
            "3.11",
            "--extra",
            "dev",
            "--extra",
            "notebook",
        ],
        cwd=ROOT,
        check=True,
        env=dict(os.environ, UV_PROJECT_ENVIRONMENT=str(ROOT / ".venv")),
    )
    PROJECT_PYTHON = str(ROOT / ".venv/bin/python")
    os.environ.setdefault("MINARI_DATASETS_PATH", "/content/minari-datasets")
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
    PROJECT_PYTHON = sys.executable
os.environ.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", MPLBACKEND="Agg")
runtime = runpy.run_path(str(ROOT / "scripts/notebook_runtime.py"))
project_kernel = runtime["register_project_magic"](ROOT, PROJECT_PYTHON)
host = {
    "frontend": "Google Colab" if IN_COLAB else "local Jupyter",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
}
project_kernel.execute("COLAB_HOST = " + repr(host))
print("Project kernel ready; Colab host Python:", sys.version.split()[0])
print("Project interpreter:", PROJECT_PYTHON)

In [ ]:
%%project
from iql_project import __version__
from iql_project.cli import main
from iql_project.config import ProjectConfig, load_config

print("Package version:", __version__)
print("Repo root:", ROOT)
assert main(["check"]) == 0, "iql-project check failed"

config_path = ROOT / "configs" / "halfcheetah.toml"
config = load_config(config_path) if config_path.exists() else ProjectConfig()
print(config)
print(
    "Convention: beta := inverse_temperature =",
    config.inverse_temperature,
    "| expectile =",
    config.expectile,
    "| max_weight =",
    config.max_weight,
)

import sys, platform, subprocess, importlib.metadata as md
import minari
import torch
torch.set_num_threads(1)
print("Project Python:", sys.version)
print("Project platform:", platform.platform())
print("Locked versions:", {p: md.version(p) for p in ["numpy", "torch", "gymnasium", "mujoco", "minari"]})
subprocess.run(["uv", "pip", "check", "--python", sys.executable], check=True)
minari.load_dataset(config.dataset_id, download=True)
print("Dataset cache ready.")

## 3. Environment — HalfCheetah-v5

HalfCheetah-v5 observes 17 continuous positions/velocities and controls six joint torques in `[-1, 1]`. The goal is forward motion with low control effort; the default reward is horizontal velocity minus $0.1$ times the squared action norm. The dataset records a 1,000-step horizon. Time-limit **truncations** end episodes while preserving the Q bootstrap.

Nikita Shankin implements `make_evaluation_env(config, render_mode=None)` from the dataset collection EnvSpec (MuJoCo **3.2.3** matches collection; see `docs/SIMULATOR_COMPATIBILITY.md` and `docs/ENVIRONMENT_DATASET.md`).

Setup downloads the dataset before recovery. Environment/data failures stop Run all; they are never reported as successful skips.

In [ ]:
%%project
from iql_project.environment import make_evaluation_env
env = make_evaluation_env(config, render_mode=None)
try:
    print("observation_space:", env.observation_space)
    print("action_space:", env.action_space)
    obs, info = env.reset(seed=config.seed)
    assert obs.shape == (17,)
    print("reset obs shape:", obs.shape)
finally:
    env.close()

## 4. Offline dataset — Minari medium

`mujoco/halfcheetah/medium-v0` contains 1,000 episodes / 1,000,000 transitions. Nikita Shankin validated: untransformed reward mean/std ≈ 12.09 / 4.70; episode return mean/std ≈ 12,089 / 3,009; every episode has 1,000 steps; zero true terminations and 1,000 truncations. These are **dataset** statistics, not trained-policy results.

Loader contract (`docs/INTERFACES.md`):

- float32 batches; observations `(B, 17)`, actions `(B, 6)`, rewards `(B, 1)`
- bool end flags `(B, 1)`
- **termination** disables Q bootstrap; **truncation alone does not**
- observation mean/std fitted on offline data only (std floored at `1e-3`)

`load_offline_dataset(config)` defaults to `download=False`.

In [ ]:
%%project
import numpy as np
from iql_project.dataset import load_offline_dataset
dataset = load_offline_dataset(config, download=False)
batch = dataset.sample(config.batch_size, np.random.default_rng(config.seed))
assert len(dataset) == 1_000_000
assert batch.observations.shape == (256, 17) and batch.actions.shape == (256, 6)
print("transitions:", len(dataset))
print("batch shapes:", batch.observations.shape, batch.actions.shape)
print("statistics:", dataset.metadata.get("statistics", dataset.metadata))

## 5. What is IQL?

Paper: Kostrikov, Nair, Levine — *Offline Reinforcement Learning with Implicit Q-Learning* (ICLR 2022).

### Motivation

Offline Q-learning that takes $\max_{a'} Q(s', a')$ queries actions that may be absent from the dataset. Those OOD actions can receive inflated values and ruin the policy.

IQL instead:

1. Fits $V(s)$ to **dataset actions only**, via **expectile regression** toward upper expectiles of $Q(s,a)$.
2. Fits $Q(s,a)$ with a **V-backed** target (no max over unseen actions).
3. Extracts a policy by **advantage-weighted regression** on dataset actions.

### The three losses (project convention)

Let $\tau =$ `expectile` (default $0.7$) and $\beta =$ **`inverse_temperature`** (default $3.0$).
In this repository **beta is inverse temperature**, not temperature: larger $\beta$ concentrates more on high-advantage actions.

**1. Value / expectile loss**

$$
u = Q(s,a) - V(s), \qquad
w_\tau(u) = \begin{cases}\tau & u \ge 0 \\ 1-\tau & u < 0\end{cases}, \qquad
\mathcal{L}_V = \mathbb{E}\big[w_\tau(u)\, u^2\big]
$$

**2. Q loss with terminated mask**

$$
y = r + \gamma\,(1-\mathbb{1}_{\text{terminated}})\, V(s'), \qquad
\mathcal{L}_Q = \mathbb{E}\big[(Q(s,a) - y)^2\big]
$$

Truncation must **not** zero the bootstrap. Only true termination does.

**3. Policy loss with capped advantage weight**

$$
A(s,a) = Q(s,a) - V(s), \qquad
w = \min\big(\exp(\beta A),\; w_{\max}\big), \qquad
\mathcal{L}_\pi = \mathbb{E}\big[- w \log \pi_\theta(a\mid s)\big]
$$

with $w_{\max}=$ `max_weight` (default $100$).

## 6. Manual IQL update — one numerical transition

Pedagogy only (plain arithmetic), not a second learner.

| Symbol | Value | Meaning |
|---|---|---|
| $Q(s,a)$ | $8.0$ | current Q prediction |
| $V(s)$ | $5.0$ | current V at $s$ |
| $V(s')$ | $6.0$ | current V at next state |
| $r$ | $1.0$ | reward |
| $\gamma$ | $0.99$ | `config.discount` |
| $\tau$ | $0.7$ | expectile |
| $\beta$ | $3.0$ | **inverse temperature** |
| $w_{\max}$ | $100$ | policy weight cap |

In [ ]:
%%project
import math

Q_sa = 8.0
V_s = 5.0
V_s_next = 6.0
reward = 1.0
gamma = config.discount
tau = config.expectile
beta = config.inverse_temperature  # inverse temperature, NOT temperature
max_weight = config.max_weight

# 1) Expectile weight for V
u = Q_sa - V_s
expectile_weight = tau if u >= 0 else (1.0 - tau)
v_loss_term = expectile_weight * (u**2)

# 2) Masked Q targets
y_continue = reward + gamma * (1.0 - 0.0) * V_s_next  # terminated=False
y_terminal = reward + gamma * (1.0 - 1.0) * V_s_next  # terminated=True

# 3) Capped policy weight
advantage = Q_sa - V_s
raw_weight = math.exp(beta * advantage)
policy_weight = min(raw_weight, max_weight)

print("residual u = Q - V =", u)
print("expectile weight w_tau =", expectile_weight)
print("V loss term w_tau * u^2 =", v_loss_term)
print("Q target (unterminated) y =", y_continue)
print("Q target (terminated) y =", y_terminal)
print("advantage A =", advantage)
print("raw exp(beta * A) =", raw_weight)
print("capped policy weight =", policy_weight)
print("beta convention: inverse_temperature =", beta)

assert abs(v_loss_term - 6.3) < 1e-9
assert abs(y_continue - 6.94) < 1e-9
assert abs(y_terminal - 1.0) < 1e-9
assert policy_weight == max_weight
print("Manual example checks passed.")

### Reading the numbers

- Residual $u=3>0$ → expectile weight $\tau=0.7$. V is pulled toward this better-than-$V$ dataset action.
- Unterminated target $y=1+0.99\cdot 6=6.94$. If **terminated**, bootstrap is masked and $y=r=1$.
- Advantage $A=3$ with $\beta=3$ gives $\mathrm{e}^{9}\approx 8103$, then **capped** to $100$.

Bad-action contrast ($Q=3$, $V=5$): $A=-2$, $\mathrm{e}^{-6}\approx 0.0025$ → almost no imitation.

## 7. Package surface (Leo Vesin) — agreed network factory

Public API is `build_networks(config, info) -> dict[str, nn.Module]` with keys `q1`, `q2`, `value`, `policy`, `target_q1` and `target_q2` (`docs/IQL.md`). Do **not** import a separate `QNetwork` / `VNetwork` / `PolicyNetwork` API from the notebook.

Losses, the tanh-Gaussian likelihood, checkpoint fields and resume are documented in `docs/IQL.md`. Evaluation reloads preprocessing with `dataset_info_from_checkpoint` before constructing `IQLAgent`.

In [ ]:
%%project
from iql_project.iql import IQLAgent
from iql_project.networks import build_networks
nets = build_networks(config, dataset.info)
print("networks:", sorted(nets))

## 8. Training protocol (Leo Vesin and Ruslan Nasibullin)

1. Load config + offline dataset (no env interaction for learning).
2. For `total_steps` updates: sample batch → `agent.update(batch)` → log JSONL.
3. Checkpoint under `checkpoints/<run_id>/`.
4. Write `results/<run_id>/manifest.json` with git commit, seeds, hardware, checksums.

Defaults: `batch_size=256`, `total_steps=500000`, `seed=0`, device `cpu` unless a CUDA lock is documented. Run all below executes a separate 1,000-update smoke run; use the CLI for the full schedule.

In [ ]:
%%project
import json, uuid
from dataclasses import replace
from iql_project.train import train
smoke_dir = ROOT / "results" / ("notebook-smoke-" + uuid.uuid4().hex[:8])
smoke_config = replace(config, total_steps=1000, checkpoint_interval=500)
smoke_checkpoint = train(smoke_config, run_dir=smoke_dir)
metrics = [json.loads(line) for line in (smoke_dir / "training.jsonl").read_text().splitlines()]
assert [row["step"] for row in metrics] == list(range(1, 1001))
print("Fresh offline updates:", len(metrics))
print("Saved checkpoint:", smoke_checkpoint)
print("This 1,000-update smoke run is separate from the published 100k experiments.")

## 9. Results and evaluation (Ruslan Nasibullin)

Evaluation uses the recovered Minari environment, saved checkpoint normalization and deterministic actions. Every checkpoint and the uniform random baseline use evaluation seeds **10000–10009**, separately for training seeds **0, 1, 2**. Protocol details: `docs/EVALUATION.md` and `docs/evidence/person-4/README.md`.

### Measured 100k-update protocol (raw returns)

Ruslan Nasibullin ran **100,000 offline updates** (not the default 500k), `batch_size=256`, checkpoints every 25k, CPU PyTorch on Apple M3 Pro. Training wall times ≈ **322 / 326 / 330 s** for seeds 0 / 1 / 2 (updates only).

| Policy / checkpoint | Mean return | Spread | Notes |
|---|---:|---:|---|
| Random baseline | **−287.40** | ±56.68 (episode pop. std) | same 10 eval seeds |
| IQL seed 0 @ 100k | **1964.28** | ±2164.35 | non-monotonic (had 3598.90 @ 50k) |
| IQL seed 1 @ 100k | **3475.73** | ±2548.32 | |
| IQL seed 2 @ 100k | **3099.20** | ±2128.01 | |
| IQL across seeds @ 100k | **2846.41** | ±786.80 (sample std of run means) | 3 training seeds |

No D4RL normalized score is claimed. Positive average return beats random, but **episode variance is large** and demos do not show robust running.

Generate local artifacts with:

```text
python scripts/run_experiments.py --steps 100000 --checkpoint-every 25000 --run-prefix person4_100k --video
```

Or load the committed evidence / release bundle below.


### Platform-scoped reproducibility

The table above is the **original macOS/Apple M3 Pro experiment**, not a promise
that all platforms return identical numbers. The same checkpoint bytes on Colab
Linux x86_64 previously returned **2187.95 ± 480.18** across run means, with random
baseline **−302.78**. The next cell produces fresh measurements and compares every
episode against a second replay on this platform. Both results remain separately
labeled; the original report is preserved.

MuJoCo guarantees exact reproducibility within the same version/architecture;
small floating-point differences can grow in contact simulation across platforms.
[Official reproducibility documentation](https://mujoco.readthedocs.io/en/3.7.0/computation/index.html#reproducibility).
This is consistent with the observed platform difference, but no specific compiler
instruction has been identified as its cause. See `docs/REPRODUCIBILITY.md`.

In [ ]:
%%project
import subprocess, uuid
REPLAY_DIR = ROOT / "results" / ("notebook-replay-" + uuid.uuid4().hex[:8])
subprocess.run([sys.executable, "scripts/replay_published.py", "--output-dir", str(REPLAY_DIR), "--video"], cwd=ROOT, check=True)
repeatability = json.loads((REPLAY_DIR / "repeatability.json").read_text())
assert repeatability["same_platform_exact_repeat"] and repeatability["episode_records_compared"] == 60
print("This platform:", repeatability["platform"])
print("Fresh replay result:", repeatability["this_platform_final"])

In [ ]:
%%project
import json
from pathlib import Path

from IPython.display import Image, display

from iql_project.evaluate import evaluate, record_video
from iql_project.reporting import evaluate_checkpoint, summarize_runs

print("Evaluation API:", evaluate, record_video, evaluate_checkpoint, summarize_runs)

# Prefer local aggregate artifacts; else committed Ruslan Nasibullin evidence.
CANDIDATES = [
    REPLAY_DIR / "aggregate",
    ROOT / "docs/evidence/person-4",
]
results_dir = next((path for path in CANDIDATES if (path / "summary.json").exists()), None)
if results_dir is None:
    print("No measured summary.json found under results/ or docs/evidence/person-4/.")
else:
    measured = json.loads((results_dir / "summary.json").read_text(encoding="utf-8"))
    print("Loaded:", results_dir)
    print("Protocol met:", measured.get("minimum_seed_episode_protocol_met"))
    print("Across training seeds (run means at each checkpoint):")
    for checkpoint_result in measured.get("across_runs", []):
        step = checkpoint_result["checkpoint_step"]
        mean = checkpoint_result["mean_of_run_means"]
        std = checkpoint_result["std_sample_of_run_means"]
        print(f"  step {step:>6}: mean={mean:.2f}, sample_std={std:.2f}")

    # Final-step within-run means for IQL
    finals = [
        row
        for row in measured.get("within_run", [])
        if row.get("policy_name") == "iql"
        and row.get("checkpoint_step")
        == measured.get("final_checkpoint_steps", {}).get(row.get("run_id"))
    ]
    if finals:
        print("Final IQL within-run means:")
        for row in finals:
            print(
                f"  seed {row['training_seed']}: "
                f"{row['return_mean']:.2f} ± {row['return_std_population']:.2f}"
            )

    plot = results_dir / "evaluation_returns.png"
    if not plot.exists():
        plot = results_dir / "plots" / "evaluation_returns.png"
    if plot.exists():
        display(Image(filename=str(plot)))
    else:
        print("Plot not found next to summary.json:", plot)

## 10. Demo, limitations, Colab acceptance

### Demo

These are real deterministic episodes from the final **100k-update** checkpoints, evaluation seed **10000**. All three policies **fall or stall**; robust running has **not** been demonstrated. A return above the random baseline is insufficient evidence of stable locomotion (Ruslan Nasibullin assessment).

Published MP4s (visible floor; 480×480, 20 fps, 50 s):

- [Training seed 0](https://github.com/MedvAx-AI/iql-halfcheetah/releases/download/person4-evaluation-100k/person4_100k_seed_0_visible_floor_demo_seed_10000.mp4) — SHA-256 `42a63d4fe617edb1cd446d9bf40701f94e4e03a027b179b9321278b8476064fe`
- [Training seed 1](https://github.com/MedvAx-AI/iql-halfcheetah/releases/download/person4-evaluation-100k/person4_100k_seed_1_visible_floor_demo_seed_10000.mp4) — SHA-256 `f34883f3411e02ffac3e44d2389636bc8719d6ef01812dab37d1f4d51bac63c0`
- [Training seed 2](https://github.com/MedvAx-AI/iql-halfcheetah/releases/download/person4-evaluation-100k/person4_100k_seed_2_visible_floor_demo_seed_10000.mp4) — SHA-256 `d4c080a30e5ced91d70b14f28e0ad08faa630343b6c6d62d0a8f074f36518abb`

Release / archive: [person4-evaluation-100k](https://github.com/MedvAx-AI/iql-halfcheetah/releases/tag/person4-evaluation-100k). Provenance: `docs/evidence/person-4/publication.json`.

### Limitations

- Measured schedule is **100k** updates; default **500k** was not run.
- Learning is not monotonic (seed 0 peaks before 100k).
- Fixed-seed demos show fall/stall; do not present them as successful stable running.
- Run all downloads the dataset/checkpoints and fails on broken environment/data checks. Network is required.
- Historical Windows smoke at `30e465c` skipped data and is not final acceptance. See the live acceptance JSON below and `docs/COLAB_ACCEPTANCE.md` for the fresh Colab evidence.

### Recorded experiment provenance

| Field | Value |
|---|---|
| Implementation commit (Ruslan Nasibullin) | `8a8c1d41f5a62ba1f8ab7dfe926b8758b57c29c2` |
| Evidence on `main` | through `c452c70` (Ruslan Nasibullin publication follow-up) |
| Config base | `configs/halfcheetah.toml` (`total_steps` overridden to 100000) |
| Training seeds | `0, 1, 2` |
| Evaluation seeds | `10000–10009` |
| Hardware / runtime | Apple M3 Pro, 36 GiB; ≈322–330 s train/seed (CPU) |
| Python stack | 3.11.x · PyTorch 2.7.1 · Gymnasium 1.2.2 · MuJoCo 3.2.3 · Minari 0.5.3 |
| Local Windows smoke (not Colab) | `30e465c`; kernel ≈9.7 s / wall ≈12 s; 0 errors; Minari env/dataset SKIP |
| Colab `Run all` | See live final acceptance cell and `docs/COLAB_ACCEPTANCE.md` (actual commit, project Python and Colab host recorded) |

### Continued training — a separately measured policy

The original three-seed 100k results above remain the course experiment. The
following checkpoint continues one of those runs on Windows CPU with the same
dataset, optimizer state, preprocessing and IQL settings. Its update count and
checksum are recorded in `docs/evidence/continuation/publication.json`.

This cell measures the selected policy on this runtime, repeats the same ten
evaluation seeds exactly, then evaluates ten additional seeds. The video uses
the same predeclared seed **10000**, the unmodified environment and the full
**1,000 steps / 50 seconds**. A successful demo does not establish reliable
running on every seed; failures and platform differences are retained in
`docs/POLICY_CONTINUATION.md`. Run all replays the checkpoint; it does not
perform the long continuation training again.

The retained reference curves show all three training seeds at 100k, 150k,
200k, 300k and 500k, measured in the recorded Colab runtime. Each point uses
ten real episodes; bars show population episode SD, not a confidence interval.
Both evaluation sets were reused while inspecting candidates. These reference
curves are separate from the selected policy's newly measured scores above.
Different CPUs can produce different trajectories and returns.


In [ ]:
%%project
import runpy
continuation_publication = json.loads((ROOT / "docs/evidence/continuation/publication.json").read_text())
continuation_api = runpy.run_path(str(ROOT / "scripts/replay_published.py"))
CONTINUATION_DIR = REPLAY_DIR / "continuation"
selected = continuation_publication["checkpoint"]
continued_checkpoint = continuation_api["download_checkpoint"](
    selected["url"], selected["sha256"], CONTINUATION_DIR / "selected.pt"
)
for name, start in [("validation", 10000), ("repeat_validation", 10000), ("additional", 20000)]:
    evaluate_checkpoint(
        continued_checkpoint, run_dir=CONTINUATION_DIR / name,
        eval_episodes=10, eval_seed=start,
        video_dir=CONTINUATION_DIR / "video" if name == "validation" else None,
    )
continuation_api["verify_repeat"](
    continuation_api["read_evaluation"](CONTINUATION_DIR / "validation/evaluation.csv"),
    continuation_api["read_evaluation"](CONTINUATION_DIR / "repeat_validation/evaluation.csv"),
)
continuation_result = {
    "checkpoint": selected,
    "platform": platform.platform(),
    "same_platform_exact_repeat": True,
    "episode_records_compared": 20,
    "validation": json.loads((CONTINUATION_DIR / "validation/summary.json").read_text()),
    "additional": json.loads((CONTINUATION_DIR / "additional/summary.json").read_text()),
    "validation_csv": (CONTINUATION_DIR / "validation/evaluation.csv").read_text(),
    "additional_csv": (CONTINUATION_DIR / "additional/evaluation.csv").read_text(),
}
print("Continued checkpoint:", selected["updates"], "updates; training seed", selected["training_seed"])
print("All 20 IQL/random validation records repeat exactly on this platform.")
for name in ("validation", "additional"):
    measured = next(row for row in continuation_result[name]["within_run"] if row["policy_name"] == "iql")
    assert measured["checkpoint_step"] == selected["updates"] and measured["training_seed"] == selected["training_seed"]
    print(name, "seeds", measured["evaluation_seeds"], "mean ± episode SD:", measured["return_mean"], measured["return_std_population"])
display(Image(filename=str(CONTINUATION_DIR / "validation/plots/evaluation_returns.png")))

print("Retained reference curves: recorded Colab evaluations, not this runtime's new measurements.")
for group in ("validation", "additional"):
    print(group, "set — 3 training seeds, 5 measured checkpoints, 10 episodes per point; mean ± episode SD")
    display(Image(filename=str(ROOT / "docs/evidence/continuation" / group / "plots/evaluation_returns.png")))


In [ ]:
%%project
from IPython.display import Video
from moviepy.video.io.VideoFileClip import VideoFileClip
VIDEO_PATH = CONTINUATION_DIR / "video/demo_seed_10000.mp4"
with VideoFileClip(str(VIDEO_PATH)) as clip:
    assert clip.size == [480, 480] and abs(clip.duration - 50) < 0.2
    for t in [0, 25, 49.9]: clip.get_frame(t)
print("Continued-policy MP4 decoded: 480x480, 50 seconds, 20 fps; fixed seed 10000")
display(Video(filename=str(VIDEO_PATH), embed=True))

In [ ]:
%%project
import csv, hashlib, platform, os, zipfile
from iql_project.iql import source_commit, runtime_versions
test_env = dict(os.environ, IQL_RUN_REAL_DATA="1")
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "--basetemp", str(REPLAY_DIR / "pytest")], cwd=ROOT, env=test_env, capture_output=True, text=True)
(REPLAY_DIR / "pytest.log").write_text(tests.stdout + tests.stderr)
print(tests.stdout)
assert tests.returncode == 0, tests.stdout + tests.stderr
subprocess.run([sys.executable, "scripts/check_dataset.py", "--output", str(REPLAY_DIR / "dataset-report.json")], cwd=ROOT, check=True)
data_report = json.loads((REPLAY_DIR / "dataset-report.json").read_text())
assert data_report["data_files_sha256"]["main_data.hdf5"] == "db46045497395bc621f2b409089e495f0e51a1182e6257f2f319f7d6654f66c5"
acceptance = {"source_commit": source_commit(), "notebook_host": COLAB_HOST, "python": platform.python_version(), "platform": platform.platform(), "versions": runtime_versions(), "dataset_transitions": len(dataset), "dataset_sha256": data_report["data_files_sha256"]["main_data.hdf5"], "smoke_training_updates": len(metrics), "smoke_checkpoint_sha256": hashlib.sha256(smoke_checkpoint.read_bytes()).hexdigest(), "smoke_training_log_sha256": hashlib.sha256((smoke_dir / "training.jsonl").read_bytes()).hexdigest(), "same_platform_exact_repeat": True, "episode_records_compared": 60, "pytest_passed": True, "pytest_output": tests.stdout.strip(), "video_decoded": True, "continuation": continuation_result, "replay": repeatability, "summary": json.loads((REPLAY_DIR / "aggregate/summary.json").read_text()), "evaluation_csv": (REPLAY_DIR / "aggregate/evaluation.csv").read_text()}
(REPLAY_DIR / "acceptance.json").write_text(json.dumps(acceptance, indent=2))
with zipfile.ZipFile(REPLAY_DIR / "acceptance-evidence.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for p in REPLAY_DIR.rglob("*"):
        if p.is_file() and p.suffix in {".json", ".csv", ".log", ".png", ".mp4"} and "pytest" not in p.parts:
            bundle.write(p, p.relative_to(REPLAY_DIR))
    bundle.write(smoke_dir / "training.jsonl", "fresh-training/training.jsonl")
    bundle.write(smoke_dir / "manifest.json", "fresh-training/manifest.json")
    bundle.write(smoke_checkpoint, "fresh-training/checkpoint.pt")
print("IQL_ACCEPTANCE_JSON " + json.dumps(acceptance))
print("RUN ALL ACCEPTANCE PASSED: real data, training, checkpoint replay, video and tests")
print("Evidence:", REPLAY_DIR / "acceptance-evidence.zip")

## 11. References

1. Ilya Kostrikov, Ashvin Nair, Sergey Levine. **Offline Reinforcement Learning with Implicit Q-Learning.** ICLR 2022. https://arxiv.org/abs/2110.06169
2. Official IQL implementation (cite commit if adapting): https://github.com/ikostrikov/implicit_q_learning
3. Minari `mujoco/halfcheetah/medium-v0`: https://minari.farama.org/datasets/mujoco/halfcheetah/medium-v0/
4. Gymnasium HalfCheetah: https://gymnasium.farama.org/environments/mujoco/half_cheetah/
5. Project docs: `docs/INTERFACES.md`, `docs/ENVIRONMENT_DATASET.md`, `docs/EVALUATION.md`, `docs/IQL.md`, `IQL_5_Person_Project_Plan.md`
6. Ruslan Nasibullin measured evidence: `docs/evidence/person-4/`, release [person4-evaluation-100k](https://github.com/MedvAx-AI/iql-halfcheetah/releases/tag/person4-evaluation-100k)

No third-party IQL code is vendored here. Adapted snippets must record source commit and license in the PR.